In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

In [2]:

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
BASE_CSV_DIR = r"./../../dataFolders/MuscaChasingBeads/Global2Local"

GLOBAL_CSV_DIR = os.path.join(BASE_CSV_DIR, "global")
LOCAL_CSV_DIR  = os.path.join(BASE_CSV_DIR, "local")

os.makedirs(GLOBAL_CSV_DIR, exist_ok=True)
os.makedirs(LOCAL_CSV_DIR, exist_ok=True)

# ==========================================================
# HELPERS
# ==========================================================
def safe_norm(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = np.nan
    return v / n

def build_body_axes(center, head, left, right):
    y = safe_norm(head - center)          # forward
    x = safe_norm(right - left)           # left-right
    z = safe_norm(np.cross(x, y))         # Up (Right-handed)
    return x, y, z

def transform_global_to_local(points, R, center):
    p_rel = points - center[:, None, :]
    return np.einsum("nij,nkj->nki", R, p_rel)

# ==========================================================
# PROCESS
# ==========================================================
POINT_IDS = ['pt2','pt3','pt4','pt5','pt6','pt7']
AXES = ['X','Y','Z']
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv")))

for path in csv_files:
    fname = os.path.basename(path)
    match = re.search(r"(Trial\d+_\d+rpm)", fname)
    trial_name = match.group(1) if match else "Unknown"
    
    print(f"Calculating coordinates for: {trial_name}")
    df = pd.read_csv(path)
    frames = df["frame"].values

    # Define Body Center and Axes
    head = df[["pt2_X", "pt2_Y", "pt2_Z"]].values
    lwb  = df[["pt3_X", "pt3_Y", "pt3_Z"]].values
    rwb  = df[["pt4_X", "pt4_Y", "pt4_Z"]].values
    center = (lwb + rwb) / 2

    # Global Points Array (N, Points, 3)
    coord_cols = [f"{pid}_{ax}" for pid in POINT_IDS for ax in AXES]
    global_pts = df[coord_cols].values.reshape(len(df), len(POINT_IDS), 3)

    # Rotation Matrix
    x_ax, y_ax, z_ax = build_body_axes(center, head, lwb, rwb)
    R = np.zeros((len(df), 3, 3))
    R[:,0,:], R[:,1,:], R[:,2,:] = x_ax, y_ax, z_ax

    # Transform
    local_pts = transform_global_to_local(global_pts, R, center)

    # Save Global
    gdf = pd.DataFrame(global_pts.reshape(len(df), -1), columns=[f"{c}_global" for c in coord_cols])
    gdf.insert(0, "frame", frames)
    gdf.to_csv(os.path.join(GLOBAL_CSV_DIR, f"{trial_name}_global.csv"), index=False)

    # Save Local
    ldf = pd.DataFrame(local_pts.reshape(len(df), -1), columns=[f"{c}_local" for c in coord_cols])
    ldf.insert(0, "frame", frames)
    ldf.to_csv(os.path.join(LOCAL_CSV_DIR, f"{trial_name}_local.csv"), index=False)

print("CSVs generated successfully.")

Calculating coordinates for: Trial2_180rpm
Calculating coordinates for: Trial2_200rpm
Calculating coordinates for: Trial3_180rpm
Calculating coordinates for: Trial4_400rpm
Calculating coordinates for: Trial5_180rpm
Calculating coordinates for: Trial5_400rpm
Calculating coordinates for: Trial7_400rpm
CSVs generated successfully.
